In [1]:
from google.colab import files
files.upload()
import json
import pandas as pd

with open(r'cleaned_dataset.json') as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(df.head())

Saving cleaned_dataset.json to cleaned_dataset (4).json
    Make       Model  EngineSize FuelType  CO2
0  ACURA         ILX         2.0        Z  196
1  ACURA         ILX         2.4        Z  221
2  ACURA  ILX HYBRID         1.5        Z  136
3  ACURA     MDX 4WD         3.5        Z  255
4  ACURA     RDX AWD         3.5        Z  244


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# Encode categorical variables
if 'Make' in df.columns and 'FuelType' in df.columns and 'Model' in df.columns:
    df = pd.get_dummies(df, columns=['Make', 'FuelType', 'Model'])

# Split features and target
X = df.drop('CO2', axis=1)
y = df['CO2']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict
preds = model.predict(X_test)

# Evaluate
print("MSE:", mean_squared_error(y_test, preds))

MSE: 217.7657408358914


In [5]:
def create_emissions_predictor(model, X_train_cols, car_info):
    """
    Returns a function that predicts total CO2 emissions for a given distance
    using a fixed car specification.
    """
    # Create DataFrame for the car
    df_car = pd.DataFrame([car_info])
    df_car = pd.get_dummies(df_car)

    # Find missing columns
    missing_cols = [col for col in X_train_cols if col not in df_car.columns]
    if missing_cols:
        # Create a DataFrame of zeros for missing columns
        zeros = pd.DataFrame(0, index=df_car.index, columns=missing_cols)
        # Concatenate them all at once
        df_car = pd.concat([df_car, zeros], axis=1)

    # Reorder columns to match training set
    df_car = df_car[X_train_cols]

    # Return a function to predict for any distance
    def predict_for_distance(distance):
        co2_per_unit = model.predict(df_car)[0]
        return co2_per_unit * distance / 1000

    return predict_for_distance

In [7]:
# Example car
my_car = {
    "Make": "AUDI",
    "Model": "A4",
    "EngineSize": 2.0,
    "FuelType": "Z"
}

# Create a predictor function for this specific car
predict_my_car_emissions = create_emissions_predictor(model, X.columns, my_car)

# Now just pass different miles
print("CO2 for 100 miles in kg:", predict_my_car_emissions(100))
print("CO2 for 500 miles in kg:", predict_my_car_emissions(500))
print("CO2 for 1000 miles in kg:", predict_my_car_emissions(1000))

CO2 for 100 miles in kg: 19.485203571428574
CO2 for 500 miles in kg: 97.42601785714287
CO2 for 1000 miles in kg: 194.85203571428573
